# 🧠 Brain Tumor MRI Classifier — Refactored CLI Notebook

This is the **command-line execution** notebook for the PyTorch pipeline.
It is designed to be executed non-interactively with Jupyter/nbconvert and to remain an **orchestration-only layer** that matches `ARCHITECTURE.md`.
All training, evaluation, TensorBoard, checkpoint, Grad-CAM, and Flask logic lives in `notebooks/brain_tumor/`. 


## 0 · Install / Refresh the Project Environment

This cell installs the package in editable mode from the repository root and pulls notebook extras.
Run it once per new environment, or skip it if the environment is already prepared from the shell.


In [1]:
from pathlib import Path
import subprocess, sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else CWD.parent
REQ_FILE = REPO_ROOT / 'requirements.txt'

if REQ_FILE.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ_FILE)])

# Optional: Install package in editable mode if pyproject.toml/setup.py exists
try:
    if (REPO_ROOT / 'pyproject.toml').exists() or (REPO_ROOT / 'setup.py').exists():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
except Exception as e:
    print(f'⚠️  Editable install skipped: {e}')

print(f'✅ Environment refreshed from: {REPO_ROOT}')

✅ Environment refreshed from: /Users/mondo/work/brain-tumor-classifier-render


## 1 · Configuration

Imports every shared constant from `./config`, then creates the project directories and seeds the run.


In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config as cfg

BASE_DIR = cfg.BASE_DIR
LOG_DIR = cfg.LOG_DIR
FLASK_HOST = cfg.FLASK_HOST
FLASK_PORT = cfg.FLASK_PORT

cfg.make_dirs()

print(f'BASE_DIR  : {BASE_DIR}')
print(f'LOG_DIR   : {LOG_DIR}')
print(f'FLASK_URL : http://{FLASK_HOST}:{FLASK_PORT}')

Device   : mps
PyTorch  : 2.11.0
CUDA     : False
MPS      : True
AMP      : False
BASE_DIR : /Users/mondo/work/brain-tumor-classifier-render
TRAIN_ROOT: /Users/mondo/work/brain-tumor-classifier-render/data/brain_tumor_mri/Training
TEST_ROOT : /Users/mondo/work/brain-tumor-classifier-render/data/brain_tumor_mri/Testing


## 2 · Flask Inference App


In [5]:
import os, sys, time, subprocess, urllib.request

FLASK_URL    = f'http://{FLASK_HOST}:{FLASK_PORT}'
FLASK_HEALTH = f'{FLASK_URL}/health'
FLASK_LOG    = LOG_DIR / 'flask_server.log'
flask_script = BASE_DIR / 'app.py'

try:
    if 'flask_proc' in globals() and flask_proc and flask_proc.poll() is None:
        flask_proc.terminate(); time.sleep(1)
except Exception:
    pass

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
log_fh = open(FLASK_LOG, 'w', buffering=1)
flask_proc = subprocess.Popen(
    [sys.executable, '-u', str(flask_script)],
    stdout=log_fh, stderr=subprocess.STDOUT,
    text=True, cwd=str(flask_script.parent), env=env,
)

healthy = False
for _ in range(30):
    if flask_proc.poll() is not None:
        break
    try:
        urllib.request.urlopen(FLASK_HEALTH, timeout=1.5)
        healthy = True
        break
    except Exception:
        time.sleep(0.5)

if healthy:
    print(f'✅ Flask running → {FLASK_URL}')
    print(f'📝 Logs: {FLASK_LOG}')
    print('🛑 Stop later with: flask_proc.terminate()')
else:
    print('❌ Flask did not start. Check logs:')
    print(FLASK_LOG.read_text()[-3000:])


✅ Flask running → http://127.0.0.1:5001
📝 Logs: /Users/mondo/work/brain-tumor-classifier-render/logs/flask_server.log
🛑 Stop later with: flask_proc.terminate()


## 3 · Optional · Shutdown

In [ ]:
ENABLE_FLASK_STOP = False
if ENABLE_FLASK_STOP and 'flask_proc' in globals() and flask_proc and flask_proc.poll() is None:
    import time
    flask_proc.terminate()
    time.sleep(1)
    print('Flask stopped.')


---

This notebook is intentionally thin. If you need to change training logic, update the package modules under `notebooks/brain_tumor/`, not the notebook cells. See `ARCHITECTURE.md` for the module ownership map.
